# End-to-End Benchmark (1000/2500/5000 x 50/100/200)

This notebook reruns the stronger benchmark configuration:

- stroke counts: **1000, 2500, 5000**
- optimization steps: **50, 100, 200**
- content images: first 10 from `images/content`
- style images: first 3 from `images/style_manifest.csv`

Generation and deception are separated:
1. generate images and metrics (deception columns blank)
2. evaluate deception from saved final images
3. aggregate and plot benchmark curves

Only final stylized outputs are saved (`brushstroke_result.png`).

In [1]:
import csv
import os
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.optim as optim
from IPython.display import display
from torchvision.utils import save_image

from deception_score import compute_deception_rate, get_artist_labels, get_deception_paths, load_deception_model
from losses import StyleTransferLosses, total_variation_loss, curvature_loss
from renderer import BrushStrokeRenderer
from utils import image_loader, pick_device

In [2]:
ROOT = Path.cwd()
CONTENT_DIR = ROOT / "images" / "content"
STYLE_MANIFEST = ROOT / "images" / "style_manifest.csv"
OUT_DIR = ROOT / "results" / "experiments_1000_2500_5000_50_100_200_notebook"
VGG_WEIGHTS = ROOT / "vgg_weights" / "vgg19_weights_normalized.h5"

IMG_SIZE = 512
NUM_STROKES_LIST = [1000, 2500, 5000]
STEPS_LIST = [50, 100, 200]

SAMPLES_PER_CURVE = 10
BRUSHES_PER_PIXEL = 20
LENGTH_SCALE = 1.1
WIDTH_SCALE = 0.1
CANVAS_COLOR = "gray"

CONTENT_WEIGHT = 1.0
STYLE_WEIGHT = 3.0
TV_WEIGHT = 0.008
CURV_WEIGHT = 4.0
LR_GEOM = 1e-1
LR_COLOR = 1e-2

torch.manual_seed(42)
device = pick_device()
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Device:", device)
print("Output dir:", OUT_DIR)

Root: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer
Device: mps
Output dir: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer/results/experiments_1000_2500_5000_50_100_200_notebook


In [3]:
content_files = sorted([
    p for p in CONTENT_DIR.iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
])[:10]
style_df = pd.read_csv(STYLE_MANIFEST).head(3)

print("#content images:", len(content_files))
print("#styles:", len(style_df))
display(style_df)

assert len(content_files) == 10, f"Expected 10 content images, got {len(content_files)}"
assert len(style_df) == 3, f"Expected 3 styles, got {len(style_df)}"

#content images: 10
#styles: 3


,style_path,artist_slug
0,images/style/starry_night.jpg,vincent-van-gogh
1,images/style/water-lilies-monet.jpg,claude-monet
2,images/style/still-life-cezanne.jpg,paul-cezanne


In [4]:
def run_one_pair(content_path: Path, style_path: Path, num_strokes: int, steps: int):
    content_img = image_loader(str(content_path), IMG_SIZE, device)
    style_img = image_loader(str(style_path), 224, device)
    _, _, H, W = content_img.shape

    vgg_loss = StyleTransferLosses(
        str(VGG_WEIGHTS),
        content_img,
        style_img,
        ["conv4_2", "conv5_2"],
        ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"],
        scale_by_y=True,
    ).to(device).eval()

    content_np = content_img[0].permute(1, 2, 0).cpu().numpy()
    renderer = BrushStrokeRenderer(
        H,
        W,
        num_strokes=num_strokes,
        samples_per_curve=SAMPLES_PER_CURVE,
        strokes_per_pixel=BRUSHES_PER_PIXEL,
        canvas_color=CANVAS_COLOR,
        length_scale=LENGTH_SCALE,
        width_scale=WIDTH_SCALE,
        content_img=content_np,
    ).to(device)

    optim_geom = optim.Adam(
        [renderer.location, renderer.curve_s, renderer.curve_e, renderer.curve_c, renderer.width],
        lr=LR_GEOM,
    )
    optim_color = optim.Adam([renderer.color], lr=LR_COLOR)

    curves = {"content": [], "style": [], "tv": [], "curvature": [], "total": []}

    for _ in range(steps):
        optim_geom.zero_grad()
        optim_color.zero_grad()

        canvas = renderer()
        canvas_img = canvas.unsqueeze(0).permute(0, 3, 1, 2).contiguous()

        content_loss, style_loss = vgg_loss(canvas_img)
        content_loss = content_loss * CONTENT_WEIGHT
        style_loss = style_loss * STYLE_WEIGHT
        tv_loss = TV_WEIGHT * total_variation_loss(renderer.location, renderer.curve_s, renderer.curve_e, K=10)
        curv_loss = CURV_WEIGHT * curvature_loss(renderer.curve_s, renderer.curve_e, renderer.curve_c)
        total_loss = content_loss + style_loss + tv_loss + curv_loss

        total_loss.backward(
            inputs=[renderer.location, renderer.curve_s, renderer.curve_e, renderer.curve_c, renderer.width],
            retain_graph=True,
        )
        optim_geom.step()

        style_loss.backward(inputs=[renderer.color])
        optim_color.step()

        curves["content"].append(float(content_loss.item()))
        curves["style"].append(float(style_loss.item()))
        curves["tv"].append(float(tv_loss.item()))
        curves["curvature"].append(float(curv_loss.item()))
        curves["total"].append(float(total_loss.item()))

    with torch.no_grad():
        final_canvas = renderer()
        final_img = final_canvas.unsqueeze(0).permute(0, 3, 1, 2).contiguous()

    mse = float(torch.mean((final_img - content_img) ** 2).item())
    return final_img.detach(), mse, curves

In [5]:
metrics_path = OUT_DIR / "metrics.csv"
fieldnames = [
    "content_image", "style_image", "target_artist", "num_strokes", "steps", "runtime_sec",
    "content_loss", "style_loss", "tv_loss", "curvature_loss", "total_loss", "reconstruction_mse",
    "deception_rate", "deception_correct", "deception_total", "output_dir",
]

with metrics_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for content_path in content_files:
        for _, style_row in style_df.iterrows():
            style_path = Path(style_row["style_path"])
            if not style_path.is_absolute():
                style_path = (ROOT / style_path).resolve()
            target_artist = style_row["artist_slug"]
            if not style_path.exists():
                print("Skipping missing style image:", style_path)
                continue

            for n in NUM_STROKES_LIST:
                for s in STEPS_LIST:
                    run_name = f"{content_path.stem}__{style_path.stem}__n{n}__s{s}"
                    run_dir = OUT_DIR / run_name
                    run_dir.mkdir(parents=True, exist_ok=True)

                    t0 = time.time()
                    final_img, mse, curves = run_one_pair(content_path, style_path, n, s)
                    runtime_sec = time.time() - t0

                    # Final image only (no intermediate step snapshots).
                    save_image(final_img, run_dir / "brushstroke_result.png")

                    # Save training curves for comparison against reference trends.
                    plt.figure(figsize=(10, 6))
                    for key, vals in curves.items():
                        plt.plot(vals, label=key)
                    plt.xlabel("Step")
                    plt.ylabel("Loss")
                    plt.title(f"Training Curves: {run_name}")
                    plt.legend()
                    plt.grid(alpha=0.3)
                    plt.tight_layout()
                    plt.savefig(run_dir / "training_loss_curves.png", dpi=150)
                    plt.close()

                    writer.writerow({
                        "content_image": str(content_path),
                        "style_image": str(style_path),
                        "target_artist": target_artist,
                        "num_strokes": n,
                        "steps": s,
                        "runtime_sec": runtime_sec,
                        "content_loss": curves["content"][-1],
                        "style_loss": curves["style"][-1],
                        "tv_loss": curves["tv"][-1],
                        "curvature_loss": curves["curvature"][-1],
                        "total_loss": curves["total"][-1],
                        "reconstruction_mse": mse,
                        "deception_rate": "",
                        "deception_correct": "",
                        "deception_total": "",
                        "output_dir": str(run_dir),
                    })
                    f.flush()
                    print("Generated:", run_name)

print("Generation complete:", metrics_path)

VGG19 weights loaded.
Generated: car__starry_night__n1000__s50
VGG19 weights loaded.
Generated: car__starry_night__n1000__s100
VGG19 weights loaded.
Generated: car__starry_night__n1000__s200
VGG19 weights loaded.
Generated: car__starry_night__n2500__s50
VGG19 weights loaded.
Generated: car__starry_night__n2500__s100
VGG19 weights loaded.
Generated: car__starry_night__n2500__s200
VGG19 weights loaded.
Generated: car__starry_night__n5000__s50
VGG19 weights loaded.
Generated: car__starry_night__n5000__s100
VGG19 weights loaded.
Generated: car__starry_night__n5000__s200
VGG19 weights loaded.
Generated: car__water-lilies-monet__n1000__s50
VGG19 weights loaded.
Generated: car__water-lilies-monet__n1000__s100
VGG19 weights loaded.
Generated: car__water-lilies-monet__n1000__s200
VGG19 weights loaded.
Generated: car__water-lilies-monet__n2500__s50
VGG19 weights loaded.
Generated: car__water-lilies-monet__n2500__s100
VGG19 weights loaded.
Generated: car__water-lilies-monet__n2500__s200
VGG19 wei

In [6]:
metrics = pd.read_csv(metrics_path)

try:
    _, split_hdf5_path, _ = get_deception_paths()
    deception_model = load_deception_model()
    artist_labels = get_artist_labels(split_hdf5_path)
    print(f"Deception model ready with {len(artist_labels)} artist slugs")

    for i, row in metrics.iterrows():
        pred_img = Path(row["output_dir"]) / "brushstroke_result.png"
        if not pred_img.exists():
            continue
        try:
            rate, details = compute_deception_rate(
                deception_model,
                [str(pred_img)],
                target_artist=row["target_artist"],
                split_hdf5_path=split_hdf5_path,
                artist_labels=artist_labels,
            )
            metrics.at[i, "deception_rate"] = float(rate)
            metrics.at[i, "deception_correct"] = int(details["correct"])
            metrics.at[i, "deception_total"] = int(details["total"])
        except ValueError as err:
            print("Deception skip:", pred_img.parent.name, "->", err)
except FileNotFoundError:
    print("Deception assets not found. Leaving deception columns empty.")

metrics.to_csv(metrics_path, index=False)
print("Deception stage complete and metrics updated")

Deception score model loaded from TF checkpoint: /Users/poojakrishan/Documents/Winter26/CS679/brushstroke-parameterized-style-transfer/deception_score_vgg/model.ckpt-790000
Deception model ready with 689 artist slugs
Deception stage complete and metrics updated


In [7]:
metrics = pd.read_csv(metrics_path)
for c in ["num_strokes", "steps", "runtime_sec", "reconstruction_mse", "total_loss", "deception_rate"]:
    metrics[c] = pd.to_numeric(metrics[c], errors="coerce")

summary = (
    metrics.groupby(["num_strokes", "steps"], as_index=False)
    .agg(
        mean_runtime_sec=("runtime_sec", "mean"),
        mean_reconstruction_mse=("reconstruction_mse", "mean"),
        mean_total_loss=("total_loss", "mean"),
        mean_deception_rate=("deception_rate", "mean"),
        num_runs=("output_dir", "count"),
    )
    .sort_values(["num_strokes", "steps"])
)
summary.to_csv(OUT_DIR / "aggregate_summary.csv", index=False)

curve = metrics.groupby("num_strokes", as_index=False)["reconstruction_mse"].mean().sort_values("num_strokes")
plt.figure(figsize=(6, 4))
plt.plot(curve["num_strokes"], curve["reconstruction_mse"], marker="o")
plt.xlabel("Number of strokes")
plt.ylabel("Mean reconstruction MSE")
plt.title("Reconstruction MSE vs Stroke Count")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "mse_vs_strokes.png", dpi=150)
plt.close()

curve = metrics.groupby("num_strokes", as_index=False)["deception_rate"].mean().sort_values("num_strokes")
plt.figure(figsize=(6, 4))
plt.plot(curve["num_strokes"], curve["deception_rate"], marker="o")
plt.xlabel("Number of strokes")
plt.ylabel("Mean deception rate")
plt.title("Deception Rate vs Stroke Count")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "deception_vs_strokes.png", dpi=150)
plt.close()

curve = metrics.groupby("steps", as_index=False)["runtime_sec"].mean().sort_values("steps")
plt.figure(figsize=(6, 4))
plt.plot(curve["steps"], curve["runtime_sec"], marker="o")
plt.xlabel("Optimization steps")
plt.ylabel("Mean runtime (sec)")
plt.title("Runtime vs Optimization Steps")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "runtime_vs_steps.png", dpi=150)
plt.close()

expected = len(content_files) * len(style_df) * len(NUM_STROKES_LIST) * len(STEPS_LIST)
print("Rows:", len(metrics), "Expected:", expected)
if len(metrics) != expected:
    print("Warning: row count mismatch. Check skipped/missing files.")

display(summary)
display(metrics.head(5))

Rows: 270 Expected: 270


,num_strokes,steps,mean_runtime_sec,mean_reconstruction_mse,mean_total_loss,mean_deception_rate,num_runs
0,1000,50,52.293523,0.034952,6634.693477,0.233333,30
1,1000,100,79.845617,0.039437,2953.825627,0.233333,30
2,1000,200,146.842597,0.045712,796.139553,0.233333,30
3,2500,50,75.886370,0.034689,715.359140,0.266667,30
4,2500,100,128.109159,0.038146,230.502539,0.300000,30
5,2500,200,231.577503,0.043007,69.298854,0.300000,30
6,5000,50,90.483081,0.039579,95.493523,0.266667,30
7,5000,100,152.004606,0.042213,31.649789,0.300000,30
8,5000,200,283.830847,0.045707,14.852606,0.300000,30


,content_image,style_image,target_artist,num_strokes,steps,runtime_sec,content_loss,style_loss,tv_loss,curvature_loss,total_loss,reconstruction_mse,deception_rate,deception_correct,deception_total,output_dir
0,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,1000,50,60.864661,10.927993,6.514524,20559.300781,0.354056,20577.097656,0.050445,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
1,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,1000,100,100.629821,11.401268,5.026020,9887.519531,0.426929,9904.374023,0.054965,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
2,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,1000,200,193.905660,11.827505,4.629066,2422.453857,0.501838,2439.412354,0.059133,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
3,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,2500,50,81.428888,10.540188,4.447063,2661.092529,0.369363,2676.449219,0.051436,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
4,/Users/poojakrishan/Documents/Winter26/CS679/b...,/Users/poojakrishan/Documents/Winter26/CS679/b...,vincent-van-gogh,2500,100,128.010424,10.915339,3.377320,871.652405,0.427014,886.372070,0.055030,0.0,0.0,1.0,/Users/poojakrishan/Documents/Winter26/CS679/b...
